# Classical Forecasts and Multi-Horizon Skill

**Group member:** Mark Muriithi Macharia  
**Stock:** JPM  
**Model:** SARIMAX with lagged market predictors  

This notebook fits a classical ARMAX model for one-day returns, then checks how a direct ARMAX forecast performs at 1, 5, 21 and 63 trading days. Each test block follows the shared expanding-window split.


## 1. Load the shared data and evaluation functions

The target is a log return. Predictor changes are shifted by one trading day so they were available before the target return was observed.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.statespace.sarimax import SARIMAX

from group_utils import get_splits, save_results, dm_vs_naive
from tslab import config
from tslab.data import load_market_data
from tslab.metrics import rmse

warnings.filterwarnings("ignore")
np.random.seed(config.SEED)

print("Ticker:", config.TARGET_TICKER)
print("Walk-forward setup: initial train =", config.INITIAL_TRAIN, "rows; test block =", config.TEST_SIZE, "rows")


## 2. Build the target and exogenous predictors

The response for the one-day model is the daily log return. The seven predictor series are transformed to daily changes; the interest rate uses a difference, while positive price series use log returns. Shifting these changes by one row keeps each predictor in the information set available before the forecast day.


In [ ]:
frame = load_market_data()
log_close = np.log(frame["Close"])
stock_return = log_close.diff()

predictor_changes = pd.DataFrame(index=frame.index)
for name in ["SP500", "TECH", "VIX", "USD", "BRENT", "GOLD"]:
    predictor_changes[name] = np.log(frame[name]).diff()
predictor_changes["US10Y"] = frame["US10Y"].diff()

exog = predictor_changes.shift(1)
data = pd.concat([stock_return.rename("y"), exog], axis=1).dropna()
y = data["y"]
X = data.drop(columns="y")

print(f"Rows: {len(data):,}; dates: {data.index.min().date()} to {data.index.max().date()}")

## 3. Select the ARMAX order on the first training block

A small AIC search chooses the autoregressive and moving-average orders using only the first training block. That order is kept fixed in the walk-forward evaluation.


In [ ]:
initial_splits = get_splits(len(data))
if not initial_splits:
    raise ValueError("The data has fewer rows than the configured initial training period.")
first_train, _ = initial_splits[0]
y_initial = y.iloc[first_train].to_numpy()
X_initial = X.iloc[first_train].to_numpy()

aic_rows = []
for p in range(3):
    for q in range(3):
        try:
            fitted = SARIMAX(
                y_initial, exog=X_initial, order=(p, 0, q), trend="c",
                enforce_stationarity=False, enforce_invertibility=False
            ).fit(disp=False, maxiter=200)
            aic_rows.append({"p": p, "d": 0, "q": q, "AIC": fitted.aic})
        except Exception as error:
            print(f"Skipped ARMAX({p},0,{q}): {error}")

aic_table = pd.DataFrame(aic_rows).sort_values("AIC").reset_index(drop=True)
if aic_table.empty:
    raise RuntimeError("No candidate ARMAX model converged on the initial training block.")
best_p, best_q = int(aic_table.iloc[0]["p"]), int(aic_table.iloc[0]["q"])
display(aic_table)
print(f"Selected ARMAX({best_p}, 0, {best_q})")


## 4. One-day-ahead walk-forward forecast

Each fold refits on the training rows. Within a test block, the model issues a one-step forecast, then incorporates the just-observed return before forecasting the next day.


In [ ]:
one_day_prediction = pd.Series(np.nan, index=data.index, name="y_pred")

for train_idx, test_idx in get_splits(len(data)):
    fitted = SARIMAX(
        y.iloc[train_idx].to_numpy(), exog=X.iloc[train_idx].to_numpy(),
        order=(best_p, 0, best_q), trend="c",
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False, maxiter=200)

    extended = fitted.extend(
        endog=y.iloc[test_idx].to_numpy(), exog=X.iloc[test_idx].to_numpy()
    )
    one_day_prediction.iloc[test_idx] = extended.fittedvalues

scored = one_day_prediction.notna()
y_test = y.loc[scored]
y_hat = one_day_prediction.loc[scored]
statistic, p_value, verdict = dm_vs_naive(y_test, y_hat, horizon=1)

one_day_summary = pd.DataFrame([
    {
        "Model": f"ARMAX({best_p},0,{best_q})",
        "Test rows": int(scored.sum()),
        "RMSE (return)": rmse(y_test, y_hat),
        "Naive RMSE (return)": rmse(y_test, np.zeros(len(y_test))),
        "DM statistic": statistic,
        "DM p-value": p_value,
        "Verdict vs naive": verdict,
    }
])
display(one_day_summary.round(6))

In [ ]:
armax_path = save_results("armax", y, one_day_prediction)
print("Saved one-day predictions to", armax_path)


## 5. Direct forecasts at four horizons

For each horizon, the target is `log(close[t+h] / close[t])`. The shared splitter leaves a gap of `h - 1` rows so a training label cannot overlap the first test target. Each test block is forecast as a batch; its actual target values are not fed back into the model. The ARMAX order selected above remains fixed. When the chosen order is `(0, 0, 0)`, the same model is an exogenous linear regression with white-noise errors, so its coefficients can be estimated directly.


In [ ]:
HORIZONS = [1, 5, 21, 63]
multi_rows = []

for horizon in HORIZONS:
    target_h = log_close.shift(-horizon) - log_close
    data_h = pd.concat([target_h.rename("y"), exog], axis=1).dropna()
    y_h, X_h = data_h["y"], data_h.drop(columns="y")
    splits_h = get_splits(len(data_h), horizon=horizon)
    prediction_h = pd.Series(np.nan, index=data_h.index, name="y_pred")

    for train_idx, test_idx in splits_h:
        if best_p == 0 and best_q == 0:
            train_design = np.column_stack([np.ones(len(train_idx)), X_h.iloc[train_idx].to_numpy()])
            test_design = np.column_stack([np.ones(len(test_idx)), X_h.iloc[test_idx].to_numpy()])
            coefficients, *_ = np.linalg.lstsq(train_design, y_h.iloc[train_idx].to_numpy(), rcond=None)
            prediction_h.iloc[test_idx] = test_design @ coefficients
        else:
            fitted_h = SARIMAX(
                y_h.iloc[train_idx].to_numpy(), exog=X_h.iloc[train_idx].to_numpy(),
                order=(best_p, 0, best_q), trend="c",
                enforce_stationarity=False, enforce_invertibility=False
            ).fit(disp=False, maxiter=200)
            prediction_h.iloc[test_idx] = fitted_h.get_forecast(
                steps=len(test_idx), exog=X_h.iloc[test_idx].to_numpy()
            ).predicted_mean

    valid = prediction_h.notna()
    actual_h = y_h.loc[valid]
    forecast_h = prediction_h.loc[valid]
    naive_rmse = rmse(actual_h, np.zeros(len(actual_h)))
    model_rmse = rmse(actual_h, forecast_h)
    dm_stat, dm_p, dm_label = dm_vs_naive(actual_h, forecast_h, horizon=horizon)
    multi_rows.append({
        "Horizon (trading days)": horizon,
        "Test rows": int(valid.sum()),
        "Test from": valid[valid].index.min().strftime("%Y-%m-%d"),
        "Test to": valid[valid].index.max().strftime("%Y-%m-%d"),
        "Model RMSE (return)": model_rmse,
        "Naive RMSE (return)": naive_rmse,
        "Skill": 1 - model_rmse / naive_rmse,
        "DM statistic": dm_stat,
        "DM p-value": dm_p,
        "Verdict vs naive": dm_label,
    })

multi_horizon = pd.DataFrame(multi_rows)
display(multi_horizon.round(6))


In [ ]:
results_dir = config.PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
multi_path = results_dir / "multi_horizon.csv"
multi_horizon.to_csv(multi_path, index=False, float_format="%.10g")
print("Saved horizon comparison to", multi_path)


## 6. Interpretation

The skill column compares each model RMSE with the zero-return naive forecast. Positive skill means lower RMSE than naive; negative skill means higher RMSE. The DM p-value tests whether the squared-error difference is statistically distinguishable from zero, with the horizon adjustment applied.

At 63 days there are fewer independent forecast periods, and market regimes can change during the horizon. Treat that result as less certain than the one-day comparison. Report the observed values from the table; do not describe a model as better than naive unless the comparison supports that conclusion.
